# Baseline Model

This notebook builds a simple base model CNN to run through all the training and testing process.

Dataset

In [ ]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from google.colab import drive

class FruitDataset(Dataset):
    """
    Custom dataset for fruit classification.
    Image label is inferred from filename prefix.
    """
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.image_files = [
            f for f in os.listdir(root_dir)
            if f.lower().endswith(".jpg")
        ]
        self.class_map = {
            "apple": 0,
            "banana": 1,
            "orange": 2,
            "mixed": 3
        }
        self.transform = transforms.Compose([
            transforms.Resize((64, 64)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        filename = self.image_files[idx]
        image_path = os.path.join(self.root_dir, filename)

        image = Image.open(image_path).convert("RGB")
        image = self.transform(image)

        for cls in self.class_map:
            if filename.startswith(cls):
                label = self.class_map[cls]
                break

        return image, label


**Baseline CNN**[64*64,2 convolutional layer]

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class FruitCNN(nn.Module):
    """
    Simple CNN baseline for fruit classification.
    """
    def __init__(self):
        super(FruitCNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        # Input image: 64x64
        # After two poolings: 16x16
        self.fc1 = nn.Linear(32 * 16 * 16, 128)
        self.fc2 = nn.Linear(128, 4)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x


Training & Testing [no validation data,10 epoch]

In [ ]:
import torch
import torch.optim as optim

def train_one_epoch(model, dataloader, criterion, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in dataloader:
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    return avg_loss, accuracy


def evaluate(model, dataloader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total


main

In [ ]:
# Paths
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/CA Notebook/image"

train_dir = os.path.join(BASE_DIR, "train")
test_dir  = os.path.join(BASE_DIR, "test")


# Datasets & loaders
train_dataset = FruitDataset(train_dir)
test_dataset = FruitDataset(test_dir)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Model setup
model = FruitCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
#Record training data
train_losses = []
train_accuracies = []

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer
    )
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")

# Final evaluation
test_acc = evaluate(model, test_loader)
print(f"Test Accuracy: {test_acc:.4f}")


Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1/10 | Loss: 1.3883 | Train Acc: 0.3458
Epoch 2/10 | Loss: 1.1107 | Train Acc: 0.6500
Epoch 3/10 | Loss: 0.7566 | Train Acc: 0.7708
Epoch 4/10 | Loss: 0.4930 | Train Acc: 0.8167
Epoch 5/10 | Loss: 0.3568 | Train Acc: 0.8708
Epoch 6/10 | Loss: 0.2947 | Train Acc: 0.8750
Epoch 7/10 | Loss: 0.2229 | Train Acc: 0.9208
Epoch 8/10 | Loss: 0.1705 | Train Acc: 0.9458
Epoch 9/10 | Loss: 0.1065 | Train Acc: 0.9750
Epoch 10/10 | Loss: 0.0900 | Train Acc: 0.9750
Test Accuracy: 0.8500


Training data curve

In [ ]:
import matplotlib.pyplot as plt

# Training Loss curve
plt.figure()
plt.plot(train_losses, marker='o')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.grid(True)
plt.show()

# Training Accuracy curve
plt.figure()
plt.plot(train_accuracies, marker='o')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training Accuracy Curve")
plt.grid(True)
plt.show()


confusion matrix

In [ ]:
import numpy as np
from tqdm import tqdm

def analyze_image_resolutions(image_dir, image_files):
    widths = []
    heights = []

    print("Image size is being measured....")
    for fname in tqdm(image_files):
        img_path = os.path.join(image_dir, fname)
        try:
            # Using Image.open().size only reads metadata and does not load image pixels, making it extremely fast.
            with Image.open(img_path) as img:
                w, h = img.size
                widths.append(w)
                heights.append(h)
        except Exception as e:
            print(f"Cannot read file  {fname}: {e}")

    # 1. Calculate average
    avg_width = np.mean(widths)
    avg_height = np.mean(heights)

    print(f"\nResult:")
    print(f"Average Width: {avg_width:.2f} px")
    print(f"Average Height: {avg_height:.2f} px")
    print(f"Size range: ({min(widths)}x{min(heights)}) to ({max(widths)}x{max(heights)})")

    # 2. Draw a two-dimensional distribution point map
    plt.figure(figsize=(8, 6))

    # Use alpha to control transparency; overlapping points will appear darker, reflecting distribution density.
    plt.scatter(widths, heights, alpha=0.5, c='royalblue', edgecolors='none', s=30)

    # Plot the average value (red crosshairs).
    plt.axvline(avg_width, color='red', linestyle='--', alpha=0.6, label=f'Avg Width: {avg_width:.1f}')
    plt.axhline(avg_height, color='green', linestyle='--', alpha=0.6, label=f'Avg Height: {avg_height:.1f}')

    plt.xlabel('Width (pixels)')
    plt.ylabel('Height (pixels)')
    plt.title('Image Resolution Distribution')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend()

    plt.tight_layout()
    plt.show()

analyze_image_resolutions(TRAIN_DIR, train_files)